# Task 1: Setting Up the Environment


In [ ]:
# Import Required Libraries
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, Tuple, List 
import sys
import os

# Add project root to path (notebook is in src/wp3_guidance_control/)
project_root = os.path.abspath(os.path.join(os.path.dirname(__file__ if '__file__' in globals() else ''), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
                                             
#robot constraints
from src.wp3_guidance_control.robot_constraints import NAO6Constraints

try:
    from controller import Robot, Motor, InertialUnit, Gyro, Accelerometer
except ImportError:
    print("Webots controller modules not found. Running in notebook mode.")

Webots controller modules not found. Running in notebook mode.


# Defining parameters
Define key parameters that control balance bnehaviour



In [ ]:
#implementation
class BalanceParameters:

    def __init__(self):
        # safe zone and no need for control
        # rad
        self.safe_roll = 0.10
        self.safe_pitch_forward = 0.10
        self.safe_pitch_backward = 0.12

        # balance zone - we need PID to recover the robot
        # I need to justify these values
        self.max_roll_angle = 0.25
        self.max_pitch_angle = 0.30

        # fall zone: need recovery sequence for these 
        self.roll_fall_angle = 0.6
        self.pitch_fall_angle = 0.6

        # angular velocity to help detect falls earlier
        self.max_angular_velocity = 3.7

    def is_stable(self, roll, pitch):
        """are we inside the safe zone? """

        return (
            np.abs(roll) < self.safe_roll and 
            - self.safe_pitch_backward < pitch < self.safe_pitch_forward
            )
            # I need to justify the minus sign in front of safe_pitch backward ## ahh found out its a Webots thing: positive pitch is rbot leaning forward and negative pitch is robot leaning backward
        
    def is_falling(self, roll, pitch, roll_rate, pitch_rate):
        """Detect if the robot is falling. falling if the angle or angular velocuty too large """
       
        angle_large = (
            np.abs(roll) > self.roll_fall_angle or 
            np.abs(pitch) > self.pitch_fall_angle
            )

        rate_large = (
            np.abs(roll_rate) > self.max_angular_velocity or
            np.abs(pitch_rate) > self.max_angular_velocity
        )

        return angle_large or rate_large
            
    def get_stability_state(self, roll, pitch):
        if self.is_falling(roll, pitch):
            return 'falling'
        elif not self.is_stable(roll, pitch):
            return 'balancing'
        else:
            return 'stable'

params = BalanceParameters()
print(f"check stability at roll = 0.2 and pitch = 0.3: {params.is_stable(0.2, 0.3)}")
print(f"check fall at roll = 0.8, pitch = 0.2 {params.is_falling(0.8, 0.2)}")
print(params.get_stability_state(0.2,0.3))

check stability at roll = 0.2 and pitch = 0.3: False
check fall at roll = 0.8, pitch = 0.2 False
balancing


In [ ]:
# balance adjustments

def calculate_balance_adjustments(imu_data, pid_roll, pid_pitch):
    roll = imu_data['roll']
    pitch = imu_data['pitch']

    # uprght so zero target
    roll_error = 0.0 - roll
    pitch_error = 0.0 - pitch

    # use pid to calculate the corrections
    roll_corrections = pid_roll.update(roll_error, dt)
    pitch_correction = pid_pitch.update(pitch_error, dt)

    # take the correction to the joints
    adjustments = {
        # roll


        #pitch


        #hip

    }

    for joint_name in adjustments:
        adjustments[joint_name] = float(np.clip(adjustments[joint_name], -0.2, 0.2))
    return adjustments

# Fall Recovery Sequences
Get the robot up after it falls down

In [ ]:
# test fall recovery

class FallRecovery:

    """detect the fall and create sequences that recover the fallen robot 
    """
    def __init__(self):
        """initialise fall recovery
        """
        self.recovery_phase = 0
        self.phase_time = 0.0

    def detect_fall_direction(self, roll, pitch, params):
        """detect the direction which the robot has fallen

        args: roll angle in rad
        pitch angle in rad

        returns: fall direction as front, left, right, backward or none?
        """
        #check pitch fall direction
        if pitch > params.pitch_fall_angle:
            return 'front'
        elif pitch < -params.pitch_fall_angle:
            return 'back'

        #check roll fall direction
        if roll > params.roll_fall_angle:
            return 'right'
        elif roll < -params.roll_fall_angle:
            return 'left'

        return 'none'

    def get_recovery_sequence_front(self, phase):
        """get joint positions to then recover the robot from front fall

        Args:
            phase: current recovery phase

        Returns:
            Dictionary of joint positions
        """
        if phase == 0:
            # tuck legs
            return {
                'LHipPitch':1.5,
                'RHipPitch':1.5,
                'LKneePitch':-2.0,
                'RKneePitch':-2.0,
                'LAnklePitch':0.0,
                'RAnklePitch':0.0,
            }

        elif phase == 1:
            return {
            'LHipPitch': 1.0,
            'RHipPitch': 1.0,
            'LKneePitch': -1.5,
            'RKneePitch': -1.5,
            'LAnklePitch': -0.5,
            'RAnklePitch': -0.5,
            }

        elif phase == 2:
            return {
            'LHipPitch': 0.0,
            'RHipPitch': 0.0,
            'LKneePitch': -1.0,
            'RKneePitch': -1.0,
            'LAnklePitch': -0.3,
            'RAnklePitch': -0.3,
            }

        elif phase == 3:
            return {
            'LHipPitch': -0.3,
            'RHipPitch': -0.3,
            'LKneePitch': 0.6,
            'RKneePitch': 0.6,
            'LAnklePitch': -0.3,
            'RAnklePitch': -0.3,
            }

        else:
        # Final standing pose
            return {
            'LHipPitch': -0.3,
            'RHipPitch': -0.3,
            'LKneePitch': 0.6,
            'RKneePitch': 0.6,
            'LAnklePitch': -0.3,
            'RAnklePitch': -0.3,
            }
    def get_recovery_sequence_back (self, phase):
        """recover the robot from backward fall"""
        if phase == 0:
            # tuck legs
            return {
            'LHipPitch': -1.0,
            'RHipPitch': -1.0,
            'LKneePitch':  2.0,
            'RKneePitch':  2.0,
            'LAnklePitch': 0.0,
            'RAnklePitch': 0.0,
            }

        elif phase == 1:
            return {
            'LHipPitch': 1.0,
            'RHipPitch': 1.0,
            'LKneePitch': -1.5,
            'RKneePitch': -1.5,
            'LAnklePitch': -0.5,
            'RAnklePitch': -0.5,
            }

        elif phase == 2:
            return {
            'LHipPitch': 0.0,
            'RHipPitch': 0.0,
            'LKneePitch': -1.0,
            'RKneePitch': -1.0,
            'LAnklePitch': -0.3,
            'RAnklePitch': -0.3,
            }

        elif phase == 3:
            return {
            'LHipPitch': -0.3,
            'RHipPitch': -0.3,
            'LKneePitch': 0.6,
            'RKneePitch': 0.6,
            'LAnklePitch': -0.3,
            'RAnklePitch': -0.3,
            }

        else:
        # Final standing pose
            return {
            'LHipPitch': -0.3,
            'RHipPitch': -0.3,
            'LKneePitch': 0.6,
            'RKneePitch': 0.6,
            'LAnklePitch': -0.3,
            'RAnklePitch': -0.3,
            }
    def update_recovery(self, dt, fall_direction):
        self.phase_time +=dt

        # each phase lasts 1 second
        if self.phase_time >= 1.0:
            self.recovery_phase += 1
            self_phase_time = 0.0

        # Get joint positions for current phase
        if fall_direction == 'front':
            joints = self.get_recovery_sequence_front(self.recovery_phase)
        elif fall_direction == 'back':
            joints = self.get_recovery_sequence_back(self.recovery_phase)
        else:
            joints = {}

        recovery_complete = self.recovery_phase >= 4

        return joints, recovery_complete
    
recovery = FallRecovery() #clss instance

recovery.detect_fall_direction()
recovery.update_recovery()

#15:34 17 Nov 2025
#goal: ZMP + complete fallRecovery Sequency

